# 01 - Problem Framing and Data Contract

## 1. Project objective for model v1

Build the first production candidate for batch demand forecasting using the curated Gold feature mart.

The objective of model v1 is to generate reliable **28-day ahead forecasts** that can later be operationalized in a scheduled batch inference workflow and written back to the warehouse for dashboard consumption, monitoring, and retraining evaluation.

This notebook does **not** train models. Its purpose is to lock the forecasting contract before EDA and experimentation begin.

## 2. Chosen source table

The single approved notebook source for v1 is:

`gold.gold_m5_daily_feature_mart`

This table will be treated as the canonical modeling input for notebook development.

Why this table is chosen:

- it is already at the required forecasting grain
- it contains the supervised target (`sales`)
- it includes business and external context already integrated into Gold
- it is stable, curated, and aligned with downstream production use

This means notebook development begins from the Gold mart, not from raw, staging, or partially transformed layers.

## 3. Forecast grain

Forecast grain for v1:

**daily item-store level**

Each prediction represents expected daily unit sales for one unique:

`(store_id, item_id, date)`

This grain is the right starting point because it preserves operational detail while still allowing later aggregation for dashboard and business reporting.

## 4. Target definition

Target variable for v1:

`sales`

Definition:

Daily unit sales for each item-store-date record.

Target characteristics:

- non-negative
- structured as a time-series target
- evaluated over a future forecast window rather than random held-out rows

## 5. Forecast horizon

Initial production horizon for v1:

**28 days ahead**

This is the first forecasting window the project will optimize for.

Why this horizon is selected:

- it is a practical batch forecasting horizon
- it aligns well with the M5 forecasting setup
- it is suitable for scheduled scoring in the production workflow

## 6. Train / validation / test strategy

The split strategy for v1 will be **strictly time-based**.

No random split will be used.

Planned structure:

- **training set** = earliest historical window
- **validation set** = next contiguous time block
- **test set** = most recent held-out block

The exact cutoff dates will be finalized in the next notebook after confirming date coverage, row counts, and feature completeness.

All evaluation must respect temporal ordering to avoid leakage.

## 7. Primary and secondary metrics

Primary metric for model selection:

**WMAPE**

Secondary metrics:

- **MAE**
- **RMSE**
- **Bias**

Metric policy for v1:

- **WMAPE** is the main business-facing model selection metric
- **MAE** helps interpret average absolute error in units
- **RMSE** helps detect larger forecast misses
- **Bias** helps detect systematic over-forecasting or under-forecasting

## 8. Leakage check assumptions

The modeling workflow must exclude any fields that directly or indirectly leak future information into the training process.

In later notebooks, all columns will be explicitly classified into:

- identifiers
- target
- candidate features
- leakage-risk fields
- columns requiring transformation before modeling

Important rule:

Lagged and rolling features are only valid if they are constructed strictly from information available at prediction time.

No feature should depend on future values of `sales` or any future-derived transformation.

## 9. Model candidates for v1

Candidate models for notebook comparison:

- naive baseline
- rolling mean baseline
- seasonal naive baseline
- gradient-boosted tree model as the first advanced candidate

Modeling direction for v1:

- baselines are mandatory and must be established first
- the expected first serious production candidate is a gradient-boosted tabular model
- interpretability matters, so the first advanced model should remain explainable enough for feature importance and later SHAP-based analysis

Deep learning is **not** the starting point for v1.

## 10. Data contract assumptions from the chosen source

The selected source table already provides the core fields needed for notebook-based model development, including:

- identifiers: `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`
- time fields: `d`, `date`, `wm_yr_wk`
- target: `sales`
- commercial feature: `sell_price`
- weather features
- macroeconomic features
- Google Trends features

At this stage, the Gold mart is treated as the stable feature source.

Additional modeling features such as lags, rolling statistics, calendar enrichments, and transformations will be created later in a controlled notebook and then moved into production-grade Python code.

## 11. Decision summary

Modeling contract for v1:

- **source table:** `gold.gold_m5_daily_feature_mart`
- **forecast grain:** daily item-store level
- **target:** `sales`
- **horizon:** 28 days
- **split strategy:** time-based only
- **primary metric:** WMAPE
- **secondary metrics:** MAE, RMSE, Bias
- **initial model path:** baselines first, then gradient-boosted model

This notebook establishes the modeling contract only.

EDA, feature inspection, split finalization, and model experimentation will be done in the next notebooks.

## 12. Out of scope for this notebook

This notebook does **not** perform:

- exploratory data analysis
- feature engineering
- lag creation
- model training
- MLflow logging
- model registration
- batch inference
- retraining logic

Its sole purpose is to define the modeling contract for the rest of the MLOps workflow.